# Лабораторная работа №2
## Виртуальный датчик для контроля процесса обжига в печи

Цель — оценить модели, которые восстанавливают текущее значение концентрации по минутной телеметрии и временной динамике процесса. Ячейки идут в порядке выполнения. Численные выводы печатаются автоматически по загруженным данным; текстовые интерпретации следует сверить с физикой процесса.

## 0. Импорты и настройки

Поправка на задержку задаётся явно. По умолчанию предполагается, что timestamp лабораторной записи — время получения результата; для задержки 10–15 минут берётся середина, 12 минут. Проверьте это предположение по описанию исходных данных.

In [3]:
from pathlib import Path
import time, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance
from scipy.stats import shapiro
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan
from statsmodels.stats.stattools import jarque_bera
import statsmodels.api as sm

SEED = 42
CWD = Path.cwd()
PROJECT_DIR = CWD if (CWD/'data').exists() else (CWD.parent if (CWD.parent/'data').exists() else CWD)
DATA_DIR = PROJECT_DIR / 'data' / 'raw'
REPORT_DIR = PROJECT_DIR / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
TARGET_TIMESTAMP_IS_RESULT_TIME = True
LAB_DELAY_MINUTES = 12  # поменяйте на известную задержку, либо 0, если timestamp — время отбора
MAX_ASOF_GAP_MINUTES = 2
ROLLING_WINDOWS = (5, 15, 30)
LAGS = (1, 2, 3, 5, 10, 15, 30)
RANDOM_STATE = SEED
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)
print('Папка данных:', DATA_DIR.resolve())

Папка данных: /workspace/scratch/428108f34fed/data/raw


## 1. Загрузка и первичный осмотр

Функции пытаются определить временную колонку и целевую колонку по именам/типам. При неоднозначности задайте `TIME_COL_X`, `TIME_COL_Y` и `TARGET_COL` вручную после просмотра `columns`.

In [5]:
FILES = {
    'X_train': DATA_DIR / 'data_train.csv',
    'y_train': DATA_DIR / 'target_train.csv',
    'X_test': DATA_DIR / 'data_test_small.csv',
    'y_test': DATA_DIR / 'target_test_small.csv',
}
DRIVE_IDS = {
    'X_train': '1BJ7eIQVOrnEP-v6JctC54TI_c2VlKtDf',
    'y_train': '1FpUfMWi0LcR0SxCQg9cdcWjcuezKafuL',
    'X_test': '1POwZC9gGuVGuMl4XRqw7xBND6mXCbrGs',
    'y_test': '1pcpkAP6kAhtEeGTb13nOWfKVzzUX2frm',
}

def ensure_data():
    missing = [k for k,p in FILES.items() if not p.exists()]
    if not missing:
        return
    try:
        import gdown
    except ImportError as e:
        raise FileNotFoundError(f'Не найдены {missing}. Скачайте CSV по ссылкам в README в {DATA_DIR.resolve()} или установите gdown.') from e
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for key in missing:
        print('Загрузка', FILES[key].name)
        gdown.download(id=DRIVE_IDS[key], output=str(FILES[key]), quiet=False)

ensure_data()
raw = {k: pd.read_csv(p) for k,p in FILES.items()}
for name, frame in raw.items():
    print(f'\n{name}: shape={frame.shape}')
    display(frame.head(3))
    print('Колонки:', frame.columns.tolist())


X_train: shape=(319546, 17)
Колонки: ['datetime', 'telemetry_0', 'telemetry_1', 'telemetry_2', 'telemetry_3', 'telemetry_4', 'telemetry_5', 'telemetry_6', 'telemetry_7', 'telemetry_8', 'telemetry_9', 'telemetry_10', 'telemetry_11', 'telemetry_12', 'telemetry_13', 'telemetry_14', 'telemetry_15']

y_train: shape=(1773, 2)
Колонки: ['Дата', 'target']

X_test: shape=(39943, 17)
Колонки: ['datetime', 'telemetry_0', 'telemetry_1', 'telemetry_2', 'telemetry_3', 'telemetry_4', 'telemetry_5', 'telemetry_6', 'telemetry_7', 'telemetry_8', 'telemetry_9', 'telemetry_10', 'telemetry_11', 'telemetry_12', 'telemetry_13', 'telemetry_14', 'telemetry_15']

y_test: shape=(318, 2)
Колонки: ['Дата', 'target']


,datetime,telemetry_0,telemetry_1,telemetry_2,telemetry_3,telemetry_4,telemetry_5,telemetry_6,telemetry_7,telemetry_8,telemetry_9,telemetry_10,telemetry_11,telemetry_12,telemetry_13,telemetry_14,telemetry_15
0,2019-03-01 00:01:00,0.737538,0.478367,0.292816,0.644016,0.919471,0.884814,0.754800,0.413295,0.47192,0.677801,0.687535,0.498407,NaN,NaN,NaN,NaN
1,2019-03-01 00:02:00,0.738406,0.478977,0.292816,0.648557,0.919471,0.887129,0.752977,0.413295,0.47192,0.677801,0.688926,0.498407,NaN,NaN,NaN,NaN
2,2019-03-01 00:03:00,0.738985,0.478977,0.292816,0.647164,0.919471,0.887129,0.755444,0.413295,0.47192,0.677801,0.690316,0.498407,NaN,NaN,NaN,NaN


,Дата,target
0,2019-03-13 17:00:00,0.123711
1,2019-03-13 21:00:00,0.319588
2,2019-03-14 01:00:00,0.255155


,datetime,telemetry_0,telemetry_1,telemetry_2,telemetry_3,telemetry_4,telemetry_5,telemetry_6,telemetry_7,telemetry_8,telemetry_9,telemetry_10,telemetry_11,telemetry_12,telemetry_13,telemetry_14,telemetry_15
0,2019-10-07 19:59:00,0.685456,0.409233,0.316832,0.611358,0.919214,0.800904,0.743001,0.465318,0.462082,0.728682,0.728682,0.728682,NaN,NaN,NaN,NaN
1,2019-10-07 20:00:00,0.681116,0.408937,0.318278,0.611417,0.919214,0.800904,0.742358,0.481420,0.462082,0.728682,0.728682,0.728682,0.521739,0.3,0.352273,0.55988
2,2019-10-07 20:01:00,0.677354,0.409186,0.318278,0.614155,0.919214,0.801194,0.742358,0.481420,0.462082,0.728682,0.728682,0.728682,NaN,NaN,NaN,NaN


,Дата,target
0,2019-10-07 23:00:00,0.182990
1,2019-10-08 01:00:00,0.164948
2,2019-10-08 03:00:00,0.216495


In [6]:
# При необходимости укажите точные имена столбцов. None включает автоматический поиск.
TIME_COL_X = None
TIME_COL_Y = None
TARGET_COL = None

def parse_datetime_column(df, requested=None):
    if requested is not None:
        return requested, pd.to_datetime(df[requested], errors='coerce')
    candidates = [c for c in df.columns if any(t in str(c).lower() for t in ['time','date','дата','время','timestamp'])]
    candidates += [c for c in df.columns if c not in candidates]
    best = (None, None, -1)
    for col in candidates:
        parsed = pd.to_datetime(df[col], errors='coerce')
        score = parsed.notna().mean()
        if score > best[2]: best = (col, parsed, score)
    if best[2] < .8:
        raise ValueError('Не удалось найти временную колонку. Укажите TIME_COL_X/TIME_COL_Y вручную.')
    return best[0], best[1]

def pick_target(df, time_col, requested=None):
    if requested is not None: return requested
    numeric = df.select_dtypes(include=np.number).columns.tolist()
    numeric = [c for c in numeric if c != time_col]
    if not numeric:
        other = [c for c in df.columns if c != time_col and pd.to_numeric(df[c], errors='coerce').notna().mean() > .8]
        numeric = other
    if not numeric: raise ValueError('Не найдена численная целевая колонка. Укажите TARGET_COL.')
    named = [c for c in numeric if any(t in str(c).lower() for t in ['target','value','конц','product','y'])]
    return named[0] if named else numeric[-1]

def prepare_telemetry(df, requested_time=None):
    tc, ts = parse_datetime_column(df, requested_time)
    out = df.drop(columns=[tc]).copy()
    out.index = ts
    out.index.name = 'timestamp'
    out = out.loc[out.index.notna()].sort_index()
    out = out[~out.index.duplicated(keep='last')]
    for c in out.columns: out[c] = pd.to_numeric(out[c], errors='coerce')
    out = out.select_dtypes(include=np.number)
    # Фиксируем минутную сетку: лаг в одну строку тогда соответствует одной минуте.
    out = out.resample('1min').mean()
    return out

def prepare_target(df, requested_time=None, requested_target=None):
    tc, ts = parse_datetime_column(df, requested_time)
    yc = pick_target(df, tc, requested_target)
    out = pd.DataFrame({'result_time': ts, 'target': pd.to_numeric(df[yc], errors='coerce')})
    out = out.dropna().sort_values('result_time')
    out = out.drop_duplicates('result_time', keep='last').reset_index(drop=True)
    out['sample_time'] = out['result_time'] - pd.to_timedelta(LAB_DELAY_MINUTES if TARGET_TIMESTAMP_IS_RESULT_TIME else 0, unit='min')
    return out, tc, yc

Xtr = prepare_telemetry(raw['X_train'], TIME_COL_X)
Xte = prepare_telemetry(raw['X_test'], TIME_COL_X)
ytr, detected_time_y, detected_target = prepare_target(raw['y_train'], TIME_COL_Y, TARGET_COL)
yte, _, _ = prepare_target(raw['y_test'], TIME_COL_Y, TARGET_COL)
print('Телеметрия train/test:', Xtr.shape, Xte.shape)
print('Лабораторная цель train/test:', ytr.shape, yte.shape)
print('Выбрана целевая колонка:', detected_target)
print('Период train:', Xtr.index.min(), '—', Xtr.index.max())
print('Период test:', Xte.index.min(), '—', Xte.index.max())
display(ytr.head())

Телеметрия train/test: (317998, 16) (39747, 16)
Лабораторная цель train/test: (1773, 3) (318, 3)
Выбрана целевая колонка: target
Период train: 2019-03-01 00:01:00 — 2019-10-07 19:58:00
Период test: 2019-10-07 19:59:00 — 2019-11-04 10:25:00


,result_time,target,sample_time
0,2019-03-13 17:00:00,0.123711,2019-03-13 16:48:00
1,2019-03-13 21:00:00,0.319588,2019-03-13 20:48:00
2,2019-03-14 01:00:00,0.255155,2019-03-14 00:48:00
3,2019-03-14 05:00:00,0.319588,2019-03-14 04:48:00
4,2019-03-14 09:00:00,0.159794,2019-03-14 08:48:00


## 2.1. EDA: телеметрия, пропуски, аномалии и лабораторные измерения

Визуализации строятся на временной частоте исходных данных. Одиночные экстремумы не удаляются автоматически: в производстве они могут означать реальное возмущение. Для моделей пропуски признаков будут заполняться медианой внутри обучаемого pipeline.

In [8]:
# Пропуски и описательная статистика
missing = Xtr.isna().mean().sort_values(ascending=False).rename('missing_share').to_frame()
display(missing)
display(Xtr.describe(percentiles=[.01,.05,.5,.95,.99]).T)
print('Число отрицательных/нулевых значений по телеметрии:')
display(pd.DataFrame({'negative':(Xtr<0).sum(), 'zero':(Xtr==0).sum()}))

Число отрицательных/нулевых значений по телеметрии:


,missing_share
telemetry_12,0.993406
telemetry_13,0.993406
telemetry_14,0.993406
telemetry_15,0.993406
telemetry_4,0.000418
telemetry_0,0.000053
telemetry_5,0.000053
telemetry_1,0.000050
telemetry_7,0.000050
telemetry_6,0.000050


,count,mean,std,min,1%,5%,50%,95%,99%,max
telemetry_0,317981.0,0.609986,0.276336,0.003472,0.005498,0.010706,0.738985,0.798300,0.825209,1.000000
telemetry_1,317982.0,0.379996,0.197836,0.000000,0.000000,0.000000,0.486084,0.503758,0.517802,0.610826
telemetry_2,317982.0,0.266778,0.088141,0.031828,0.038193,0.045427,0.295710,0.324355,0.436620,1.000000
telemetry_3,317982.0,0.520087,0.259315,0.000000,0.000000,0.000000,0.639477,0.686892,0.709960,1.000000
telemetry_4,317865.0,0.762025,0.325680,0.000000,0.000000,0.000000,0.919471,0.919471,0.919728,1.000000
telemetry_5,317981.0,0.695859,0.284229,0.000000,0.039061,0.044270,0.829549,0.894941,0.913459,1.000000
telemetry_6,317982.0,0.594035,0.300441,0.000000,0.000000,0.000000,0.749222,0.775287,0.785799,1.000000
telemetry_7,317982.0,0.522698,0.252907,0.020644,0.276631,0.304294,0.410817,1.000000,1.000000,1.000000
telemetry_8,317982.0,0.469586,0.023767,0.344030,0.414919,0.433727,0.467869,0.507799,0.520530,1.000000
telemetry_9,317982.0,0.535839,0.254780,0.000000,0.038104,0.041719,0.581846,0.902251,1.000000,1.000000


,negative,zero
telemetry_0,0,0
telemetry_1,0,64970
telemetry_2,0,0
telemetry_3,0,59908
telemetry_4,0,42233
telemetry_5,0,2
telemetry_6,0,59622
telemetry_7,0,0
telemetry_8,0,0
telemetry_9,0,1


In [9]:
# Обзор трендов: первые 6 сигналов (все сигналы доступны в таблицах)
cols = Xtr.columns[:min(6, len(Xtr.columns))]
fig, axes = plt.subplots(len(cols), 1, figsize=(15, 2.6*len(cols)), sharex=True)
if len(cols)==1: axes=[axes]
for ax,c in zip(axes, cols): ax.plot(Xtr.index, Xtr[c], lw=.6); ax.set_title(str(c)); ax.set_ylabel('значение')
plt.tight_layout(); plt.show()

# Корреляция телеметрических каналов
plt.figure(figsize=(12,9)); sns.heatmap(Xtr.corr(), cmap='coolwarm', center=0); plt.title('Корреляции каналов телеметрии'); plt.show()

<Figure size 1500x1560 with 6 Axes>

<Figure size 1200x900 with 2 Axes>

In [10]:
# Суточная сезонность и ориентировочная проверка выбросов (IQR, без автоматического удаления).
hour_profile = Xtr.groupby(Xtr.index.hour).median()
display(hour_profile.head())
fig, ax = plt.subplots(figsize=(12,4))
for c in Xtr.columns[:min(6,len(Xtr.columns))]: ax.plot(hour_profile.index, hour_profile[c], marker=".", label=str(c))
ax.set_xlabel("Час суток"); ax.set_ylabel("Медиана сигнала"); ax.set_title("Профиль телеметрии по часу суток"); ax.legend(ncol=2); plt.show()
q1, q3 = Xtr.quantile(.25), Xtr.quantile(.75)
iqr = q3-q1
outlier_share = ((Xtr < q1-1.5*iqr) | (Xtr > q3+1.5*iqr)).mean().rename("IQR_flag_share")
display(outlier_share.sort_values(ascending=False).to_frame())
print("IQR отмечает кандидатов для проверки; технологически реальные пики не удаляем автоматически.")

IQR отмечает кандидатов для проверки; технологически реальные пики не удаляем автоматически.


,telemetry_0,telemetry_1,telemetry_2,telemetry_3,telemetry_4,telemetry_5,telemetry_6,telemetry_7,telemetry_8,telemetry_9,telemetry_10,telemetry_11,telemetry_12,telemetry_13,telemetry_14,telemetry_15
timestamp,,,,,,,,,,,,,,,,
0,0.739563,0.487470,0.296288,0.638106,0.919214,0.832443,0.749866,0.407514,0.467580,0.580734,0.600481,0.613831,0.527950,0.51,0.443182,0.479042
1,0.738406,0.487723,0.296288,0.638116,0.919471,0.831864,0.749222,0.409579,0.468448,0.582680,0.611189,0.609381,NaN,NaN,NaN,NaN
2,0.738117,0.486000,0.296578,0.636951,0.919471,0.828971,0.749866,0.407102,0.467580,0.575866,0.610215,0.607434,0.515528,0.51,0.431818,0.485030
3,0.738406,0.485944,0.295999,0.637073,0.919471,0.828103,0.749222,0.410817,0.467869,0.564602,0.609798,0.602567,NaN,NaN,NaN,NaN
4,0.736380,0.484322,0.295999,0.637613,0.919471,0.826945,0.749222,0.413708,0.468448,0.569330,0.604931,0.605487,0.521739,0.51,0.431818,0.476048


,IQR_flag_share
telemetry_1,0.225385
telemetry_6,0.220772
telemetry_7,0.220074
telemetry_4,0.213872
telemetry_0,0.212989
telemetry_2,0.212687
telemetry_3,0.206070
telemetry_5,0.179941
telemetry_8,0.012588
telemetry_14,0.000085


<Figure size 1200x400 with 1 Axes>

In [11]:
# Целевая переменная: распределение и регулярность проб
fig, ax = plt.subplots(1,2,figsize=(14,4))
sns.histplot(ytr['target'], kde=True, ax=ax[0]); ax[0].set_title('Распределение лабораторной цели')
ax[1].plot(ytr['sample_time'], ytr['target'], marker='o', ms=3, lw=.8); ax[1].set_title('Лабораторные измерения по времени'); ax[1].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()
intervals = ytr['sample_time'].sort_values().diff().dt.total_seconds().div(60).dropna()
print(f'Частота лабораторных меток: n={len(ytr)}, медианный интервал={intervals.median():.1f} мин, P90={intervals.quantile(.9):.1f} мин')
print(f'Предполагаемая поправка времени: {LAB_DELAY_MINUTES if TARGET_TIMESTAMP_IS_RESULT_TIME else 0} мин')

Частота лабораторных меток: n=1773, медианный интервал=120.0 мин, P90=240.0 мин
Предполагаемая поправка времени: 12 мин


<Figure size 1400x400 with 2 Axes>

### Синхронизация источников и лаги

Лабораторное время результата корректируется назад на ожидаемую задержку. Затем каждому измерению сопоставляется состояние телеметрии на момент отбора через `merge_asof(direction='backward')`: данные из будущего не попадают в прогноз. Сначала изучаются лаговые корреляции; корреляция сама по себе не доказывает причинность.

In [13]:
# Частотная синхронизация и лаговая корреляция на редких метках
sync_probe = pd.merge_asof(
    ytr[['sample_time','target']].sort_values('sample_time'),
    Xtr.reset_index().rename(columns={'timestamp':'telemetry_time'}).sort_values('telemetry_time'),
    left_on='sample_time', right_on='telemetry_time', direction='backward',
    tolerance=pd.Timedelta(minutes=MAX_ASOF_GAP_MINUTES)
)
print('Доля меток с телеметрией в допуске:', round(sync_probe[Xtr.columns[0]].notna().mean(),3))
lag_rows=[]
for c in Xtr.columns:
    for lag in (0,1,3,5,10,15,30):
        shifted = Xtr[c].shift(lag)
        tmp = pd.DataFrame({'timestamp':Xtr.index, 'x':shifted.to_numpy()}).dropna().sort_values('timestamp')
        joined = pd.merge_asof(ytr[['sample_time','target']].sort_values('sample_time'), tmp,
                               left_on='sample_time', right_on='timestamp', direction='backward',
                               tolerance=pd.Timedelta(minutes=MAX_ASOF_GAP_MINUTES)).dropna()
        if len(joined)>=5:
            lag_rows.append((c,lag,joined['x'].corr(joined['target']),len(joined)))
lag_corr = pd.DataFrame(lag_rows,columns=['feature','lag_min','correlation','n']).sort_values('correlation',key=lambda s:s.abs(),ascending=False)
display(lag_corr.head(20))

Доля меток с телеметрией в допуске: 0.999


,feature,lag_min,correlation,n
0,telemetry_0,0,-0.180310,1772
1,telemetry_0,1,-0.179402,1772
4,telemetry_0,10,-0.177448,1772
2,telemetry_0,3,-0.177327,1772
3,telemetry_0,5,-0.176494,1772
5,telemetry_0,15,-0.173384,1772
6,telemetry_0,30,-0.164095,1772
26,telemetry_3,15,-0.160732,1772
27,telemetry_3,30,-0.145147,1772
25,telemetry_3,10,-0.136089,1772


### Выводы по EDA

Интерпретируйте выводы после выполнения: частоту и периоды телеметрии, долю пропусков, интервалы лабораторных проб, визуальные режимы и наиболее связанные каналы/лаги. Проверьте, разумно ли предположение о timestamp лабораторного результата. Не удаляйте выбросы без технологического обоснования.

In [15]:
print(f'EDA: {len(Xtr.columns)} численных каналов; медианная доля пропусков={Xtr.isna().mean().median():.3f}; максимум={Xtr.isna().mean().max():.3f}.')
print(f'Лабораторных наблюдений: {len(ytr)}; медианный интервал {intervals.median():.1f} минут. Синхронизация использует допуск {MAX_ASOF_GAP_MINUTES} мин и лаговую поправку {LAB_DELAY_MINUTES if TARGET_TIMESTAMP_IS_RESULT_TIME else 0} мин.')
print('Проверьте графики и таблицы выше и добавьте технологическую интерпретацию выявленных пиков и пропусков.')

EDA: 16 численных каналов; медианная доля пропусков=0.000; максимум=0.993.
Лабораторных наблюдений: 1773; медианный интервал 120.0 минут. Синхронизация использует допуск 2 мин и лаговую поправку 12 мин.
Проверьте графики и таблицы выше и добавьте технологическую интерпретацию выявленных пиков и пропусков.


## 2.2. Признаки временной динамики

Используются текущие каналы и лаги 1–30 минут, скользящие mean/std/min/max за 5/15/30 минут, а также разности и относительные изменения. Окна включают текущее время и прошлое, но никогда будущие значения. Признаки строятся отдельно для каждого временного сегмента.

In [17]:
def make_features(telemetry):
    x = telemetry.sort_index().astype(np.float32).copy()
    # Заполняем только короткие разрывы вперед по времени; остаточные пропуски обработает обучаемый imputer.
    x = x.ffill(limit=2)
    feats = {}
    for c in x.columns:
        feats[f'{c}__now'] = x[c].astype(np.float32)
        for lag in LAGS:
            feats[f'{c}__lag_{lag}m'] = x[c].shift(lag).astype(np.float32)
        for w in ROLLING_WINDOWS:
            roll = x[c].rolling(w, min_periods=max(2,w//2))
            feats[f'{c}__mean_{w}m'] = roll.mean().astype(np.float32)
            feats[f'{c}__std_{w}m'] = roll.std().astype(np.float32)
            feats[f'{c}__min_{w}m'] = roll.min().astype(np.float32)
            feats[f'{c}__max_{w}m'] = roll.max().astype(np.float32)
        for lag in (1,5,15):
            feats[f'{c}__diff_{lag}m'] = (x[c] - x[c].shift(lag)).astype(np.float32)
            feats[f'{c}__pct_{lag}m'] = x[c].pct_change(lag, fill_method=None).replace([np.inf,-np.inf],np.nan).astype(np.float32)
    return pd.DataFrame(feats, index=x.index).replace([np.inf,-np.inf], np.nan)

Ftr = make_features(Xtr)
Fte = make_features(Xte)
print('Размерность после генерации признаков:', Ftr.shape, Fte.shape)
print('Политика пропусков: короткий ffill до 2 минут; прочие пропуски — медианный imputer внутри pipeline. Выбросы не удаляются автоматически.')

Размерность после генерации признаков: (317998, 416) (39747, 416)
Политика пропусков: короткий ffill до 2 минут; прочие пропуски — медианный imputer внутри pipeline. Выбросы не удаляются автоматически.


### Выводы по признакам
Лаги отражают инерцию процесса, скользящие статистики — локальный режим и вариативность, а разности и темпы — скорость изменения. PCA не используется: число лабораторных меток ограничено, и преобразованные компоненты сложнее сопоставить с датчиками. Важность признаков далее помогает определить, какие временные окна полезны.

## Формирование обучающей выборки

Сопоставляем признаки с лабораторной меткой через последнее предыдущее/текущее состояние. Целевые метки без временной телеметрии в окне допуска исключаются. Размерность можно сократить отбором по важности после обучения; PCA не применяется, потому что ухудшает физическую интерпретируемость каналов.

In [20]:
def align_labels(features, labels, split_name):
    left = labels[['sample_time','target']].sort_values('sample_time').copy()
    right = features.reset_index().rename(columns={features.index.name or 'index':'telemetry_time'}).sort_values('telemetry_time')
    # на случай отсутствующего имени индекса
    if 'telemetry_time' not in right:
        right = features.reset_index().rename(columns={features.reset_index().columns[0]:'telemetry_time'}).sort_values('telemetry_time')
    merged = pd.merge_asof(left, right, left_on='sample_time', right_on='telemetry_time',
                           direction='backward', tolerance=pd.Timedelta(minutes=MAX_ASOF_GAP_MINUTES))
    merged = merged.dropna(subset=['telemetry_time']).reset_index(drop=True)
    merged['split'] = split_name
    return merged

train = align_labels(Ftr, ytr, 'train')
test = align_labels(Fte, yte, 'test')
feature_cols = list(Ftr.columns)
print('После синхронизации train/test:', train.shape, test.shape)
print(f'Сохранено меток: train {len(train)}/{len(ytr)}, test {len(test)}/{len(yte)}')
if len(train)<20: warnings.warn('Очень мало обучающих меток: оценки моделей будут нестабильны.')
X_train, y_train = train[feature_cols], train['target']
X_test, y_test = test[feature_cols], test['target']
if len(test)==0: raise ValueError('Нет test меток с доступной телеметрией; проверьте временные колонки, поправку задержки и MAX_ASOF_GAP_MINUTES.')

После синхронизации train/test: (1772, 420) (318, 420)
Сохранено меток: train 1772/1773, test 318/318


## 2.3. Модели и схема сравнения

1. **Ridge** — линейный регуляризованный ориентир; стандартизация обязательна из-за разных шкал датчиков.
2. **Random Forest** — нелинейные взаимодействия и устойчивость к выбросам; удобен для интерпретации важностей, но может быть тяжёлым на сотнях тысяч строк.
3. **HistGradientBoosting** — эффективный градиентный бустинг для табличных данных и нелинейностей.
4. **Наивный last value** — baseline: последняя лабораторная метка, доступная на момент прогноза.

Для статистического сопоставления дополнительно оцениваются AR-модели разных порядков на последовательности лабораторных наблюдений. Их AIC/BIC сравнимы только между AR-порядками на одной и той же обучающей последовательности; они не сравнимы напрямую с ML-моделями, использующими телеметрию.

In [22]:
models = {
    'Ridge': make_pipeline(SimpleImputer(strategy='median', keep_empty_features=True), StandardScaler(), Ridge(alpha=10.0)),
    'RandomForest': make_pipeline(SimpleImputer(strategy='median', keep_empty_features=True), RandomForestRegressor(
        n_estimators=250, min_samples_leaf=2, max_features=.7, n_jobs=1, random_state=SEED)),
    'HistGradientBoosting': make_pipeline(SimpleImputer(strategy='median', keep_empty_features=True), HistGradientBoostingRegressor(
        max_iter=250, learning_rate=.06, max_leaf_nodes=15, l2_regularization=1.0, random_state=SEED)),
}

def metric_dict(y_true, pred):
    yt, yp = np.asarray(y_true), np.asarray(pred)
    mae = mean_absolute_error(yt, yp)
    rmse = mean_squared_error(yt, yp)**.5
    nz = np.abs(yt) > 1e-12
    mape = np.mean(np.abs((yt[nz]-yp[nz])/yt[nz]))*100 if nz.any() else np.nan
    wape = np.sum(np.abs(yt-yp))/np.sum(np.abs(yt))*100 if np.sum(np.abs(yt))>0 else np.nan
    return {'MAE':mae, 'RMSE':rmse, 'MAPE_%':mape, 'WAPE_%':wape}

# Последовательный time-series CV без перемешивания.
n_splits = min(5, max(2, len(train)//10))
tscv = TimeSeriesSplit(n_splits=n_splits)
cv_rows=[]
for name, model in models.items():
    for fold,(i_tr,i_va) in enumerate(tscv.split(X_train),1):
        m=clone(model); t0=time.perf_counter(); m.fit(X_train.iloc[i_tr], y_train.iloc[i_tr]); fit_s=time.perf_counter()-t0
        p=m.predict(X_train.iloc[i_va])
        cv_rows.append({'model':name,'fold':fold,'fit_seconds':fit_s,**metric_dict(y_train.iloc[i_va],p)})
cv = pd.DataFrame(cv_rows)
cv_summary = cv.groupby('model').agg({c:['mean','std'] for c in ['MAE','RMSE','MAPE_%','WAPE_%','fit_seconds']})
display(cv_summary)
print('CV используется для выбора модели; окончательная оценка ниже выполняется один раз на test.')

CV используется для выбора модели; окончательная оценка ниже выполняется один раз на test.


/opt/codex/runtimes/codex-primary-runtime/dependencies/python/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/opt/codex/runtimes/codex-primary-runtime/dependencies/python/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


MAE                 RMSE                 MAPE_%  \
                          mean       std       mean        std        mean   
model                                                                        
HistGradientBoosting  0.072913  0.003914   0.091756   0.008829   32.655391   
RandomForest          0.071493  0.004158   0.089588   0.008467   32.055889   
Ridge                 1.693405  3.596447  27.195750  60.520794  994.357535   

                                       WAPE_%              fit_seconds  \
                              std        mean          std        mean   
model                                                                    
HistGradientBoosting     4.962120   28.304593     1.843133    1.303021   
RandomForest             5.642662   27.771116     2.212120   14.643680   
Ridge                 2141.133325  768.350584  1647.245187    0.037464   

                                
                           std  
model                           
HistGradientBoosting  0.199279  
RandomForest          9.314908  
Ridge                 0.018209

In [23]:
# Финальное обучение всех моделей и тестовая оценка
fitted={}; test_rows=[]; test_predictions={}
# baseline: последнее значение цели до текущего момента; не использует будущие метки
for name, model in models.items():
    t0=time.perf_counter(); model.fit(X_train,y_train); fit_s=time.perf_counter()-t0
    pred=model.predict(X_test)
    fitted[name]=model; test_predictions[name]=pred
    test_rows.append({'model':name,'fit_seconds':fit_s,**metric_dict(y_test,pred)})

# Persistence на лабораторных пробах с sample_time не позже времени прогноза
lab_history = ytr[['sample_time','target']].sort_values('sample_time')
persistence=[]
for t in test['sample_time']:
    hist=lab_history[lab_history['sample_time']<=t]
    persistence.append(hist['target'].iloc[-1] if len(hist) else float(y_train.median()))
test_predictions['Persistence_last_lab']=np.asarray(persistence)
test_rows.append({'model':'Persistence_last_lab','fit_seconds':0.0,**metric_dict(y_test,persistence)})
results=pd.DataFrame(test_rows).sort_values('MAE')
display(results)
results.to_csv(REPORT_DIR/'test_metrics.csv',index=False)
# Метрика направления: доля совпавших знаков изменения на соседних тестовых точках
for name,pred in test_predictions.items():
    true_dir=np.sign(np.diff(np.asarray(y_test)))
    pred_dir=np.sign(np.diff(np.asarray(pred)))
    print(f'{name}: direction accuracy={np.mean(true_dir==pred_dir):.3f} (n={len(true_dir)})')

Ridge: direction accuracy=0.555 (n=317)
RandomForest: direction accuracy=0.546 (n=317)
HistGradientBoosting: direction accuracy=0.511 (n=317)
Persistence_last_lab: direction accuracy=0.050 (n=317)


,model,fit_seconds,MAE,RMSE,MAPE_%,WAPE_%
2,HistGradientBoosting,1.322861,0.061196,0.078371,32.480378,29.182116
1,RandomForest,33.892463,0.062997,0.078050,34.787008,30.041129
0,Ridge,0.182476,0.067923,0.086614,38.444391,32.390201
3,Persistence_last_lab,0.000000,0.080999,0.110387,31.832584,38.625647


### Визуальная проверка прогноза и сравнительный вывод

In [25]:
# Выбор модели делается по train-only временной CV, а не по тестовой выборке.
best_name = cv.groupby('model')['MAE'].mean().idxmin()
best_pred = test_predictions[best_name]
best_test_row = results.loc[results['model']==best_name].iloc[0]
fig,ax=plt.subplots(figsize=(15,5))
ax.plot(test['sample_time'],y_test.to_numpy(),label='Лабораторное значение',marker='o',ms=4)
ax.plot(test['sample_time'],best_pred,label=f'Прогноз: {best_name}',marker='.',lw=1)
ax.set_title('Тестовый прогноз против лабораторных измерений'); ax.set_ylabel('Концентрация'); ax.legend(); ax.tick_params(axis='x',rotation=30)
plt.tight_layout(); plt.show()
print(f'Модель, выбранная по train CV: {best_name}; test MAE={best_test_row["MAE"]:.4g}, RMSE={best_test_row["RMSE"]:.4g}.')
print('Для выбора ориентируйтесь прежде всего на CV; тестовый результат — итоговая независимая проверка.')

Модель, выбранная по train CV: RandomForest; test MAE=0.063, RMSE=0.07805.
Для выбора ориентируйтесь прежде всего на CV; тестовый результат — итоговая независимая проверка.


<Figure size 1500x500 with 1 Axes>

### Информационные критерии AR и вычислительная эффективность

AR(p) служит статистической точкой отсчёта для последовательности редких лабораторных измерений. Поскольку интервалы проб нерегулярны, AR трактует соседние пробы как одинаковый шаг и не заменяет временную модель с правильной частотой. AIC/BIC приводятся только для ориентировочного сравнения порядков AR на одном train наборе.

In [27]:
ar_rows=[]
y_seq = y_train.astype(float).reset_index(drop=True)
for p in range(1, min(6, max(2, len(y_seq)//5))):
    try:
        fit=sm.tsa.AutoReg(y_seq,lags=p,trend='c').fit()
        ar_rows.append({'model':f'AR({p})','AIC':fit.aic,'BIC':fit.bic,'nobs':fit.nobs})
    except Exception as e: print('AR order skipped',p,repr(e))
ar_ic=pd.DataFrame(ar_rows)
display(ar_ic)
print('AIC/BIC сравниваются только внутри таблицы AR; они не являются метрикой качества прогноза ML на test.')

AIC/BIC сравниваются только внутри таблицы AR; они не являются метрикой качества прогноза ML на test.


,model,AIC,BIC,nobs
0,AR(1),-4051.743728,-4035.305829,1771
1,AR(2),-4058.661291,-4036.746352,1770
2,AR(3),-4056.064964,-4028.674116,1769
3,AR(4),-4061.181098,-4028.315473,1768
4,AR(5),-4058.271535,-4019.932266,1767


## Анализ остатков лучшей ML-модели

Проверяются среднее/разброс, нормальность (Jarque–Bera; Shapiro на подвыборке), автокорреляция (Ljung–Box), гетероскедастичность (Breusch–Pagan). Для редких наблюдений тесты имеют ограниченную мощность; графики и предметный смысл важнее формального p-value.

In [29]:
resid = np.asarray(y_test)-np.asarray(best_pred)
fig,ax=plt.subplots(1,3,figsize=(15,4))
ax[0].scatter(best_pred,resid,s=18,alpha=.7); ax[0].axhline(0,color='red',ls='--'); ax[0].set_title('Остатки vs прогноз'); ax[0].set_xlabel('прогноз'); ax[0].set_ylabel('остаток')
sns.histplot(resid,kde=True,ax=ax[1]); ax[1].set_title('Распределение остатков')
sm.qqplot(resid,line='45',ax=ax[2]); ax[2].set_title('Q-Q plot')
plt.tight_layout(); plt.show()
print('Остатки: mean=',float(np.mean(resid)),'std=',float(np.std(resid)))
if len(resid)>=8:
    print('Jarque-Bera:', jarque_bera(resid))
    print('Shapiro (до 5000 точек):', shapiro(resid[:5000]))
    print('Ljung-Box:', acorr_ljungbox(resid,lags=[min(5,max(1,len(resid)//3))],return_df=True))
    try:
        bp=het_breuschpagan(resid, sm.add_constant(np.asarray(best_pred)))
        print('Breusch-Pagan (LM, p, F, p):',bp)
    except Exception as e: print('BP не применим:',e)
else:
    print('Для формальных тестов остатков недостаточно наблюдений (нужно хотя бы 8); используйте графики.')

Остатки: mean= -0.00991274239405442 std= 0.07741797218911749
Jarque-Bera: (np.float64(32.50219474920919), np.float64(8.754635831179265e-08), np.float64(0.7569661087620947), np.float64(3.4012543717784096))
Shapiro (до 5000 точек): ShapiroResult(statistic=np.float64(0.9589981730340358), pvalue=np.float64(8.777369353767795e-08))
Ljung-Box:       lb_stat     lb_pvalue
5  125.787117  1.863218e-25
Breusch-Pagan (LM, p, F, p): (np.float64(14.030002462923647), np.float64(0.00017991691455681582), np.float64(14.585257802435287), np.float64(0.00016123766259230706))


<Figure size 1500x400 with 3 Axes>

## Важность признаков

Permutation importance вычисляется на тесте только для описательной интерпретации после фиксации модели и не используется для её повторного выбора. Взаимосвязанные лаги могут делить важность. Значения не являются причинным эффектом.

In [31]:
if best_name in fitted:
    # Ограничим число повторов для экономии времени; делаем перестановки только на выбранной модели.
    perm=permutation_importance(fitted[best_name],X_test,y_test,n_repeats=5,random_state=SEED,
                                scoring='neg_mean_absolute_error',n_jobs=1)
    importance=pd.DataFrame({'feature':feature_cols,'importance_mean':perm.importances_mean,
                             'importance_std':perm.importances_std}).sort_values('importance_mean',ascending=False)
    display(importance.head(25))
    plt.figure(figsize=(10,7)); sns.barplot(data=importance.head(20),x='importance_mean',y='feature');
    plt.title(f'Permutation importance: {best_name}'); plt.tight_layout(); plt.show()
    importance.to_csv(REPORT_DIR/'permutation_importance.csv',index=False)
else:
    print('Persistence baseline не обучается; важность признаков для него не рассчитывается.')

,feature,importance_mean,importance_std
6,telemetry_0__lag_15m,0.000379,0.000210
16,telemetry_0__mean_30m,0.000291,0.000105
7,telemetry_0__lag_30m,0.000246,0.000121
162,telemetry_6__lag_15m,0.000216,0.000058
15,telemetry_0__max_15m,0.000203,0.000127
18,telemetry_0__min_30m,0.000181,0.000042
5,telemetry_0__lag_10m,0.000178,0.000109
84,telemetry_3__lag_15m,0.000171,0.000078
94,telemetry_3__mean_30m,0.000122,0.000055
19,telemetry_0__max_30m,0.000119,0.000274


<Figure size 1000x700 with 1 Axes>

## 2.4. Итоговые аналитические выводы

Сформулируйте выводы по четырём блокам: (1) качество и состояние данных, включая смысл задержки; (2) полезность лагов/скользящих признаков и ограничение размерности; (3) сравнение моделей по CV, test, направлению, времени обучения и диагностике остатков; (4) производственная интерпретация важностей и ограничения внедрения. Не трактуйте корреляции/перестановочную важность как причинность.

In [33]:
best_row=best_test_row
print('ИТОГОВАЯ СВОДКА')
print(f'1) Использовано {len(train)} train и {len(test)} test лабораторных меток; признаков {len(feature_cols)}.')
print(f'2) Выбранная по CV модель {best_name}; test MAE={best_row.MAE:.4g}, RMSE={best_row.RMSE:.4g}, WAPE={best_row["WAPE_%"]:.2f}%.')
print('3) Проверьте согласованность test-метрик с временной CV таблицей и графиком ошибок.')
print('4) Перед производственным применением нужны технологическая валидация, проверка дрейфа датчиков и безопасный режим работы; виртуальный датчик — поддержка оператора, а не замена лабораторного контроля.')
summary={'best_model':str(best_name),'test_metrics':{k:float(v) for k,v in best_row.items() if k in ['MAE','RMSE','MAPE_%','WAPE_%']},
         'n_train_labels':int(len(train)),'n_test_labels':int(len(test)),'n_features':int(len(feature_cols)),
         'delay_minutes':int(LAB_DELAY_MINUTES if TARGET_TIMESTAMP_IS_RESULT_TIME else 0)}
(REPORT_DIR/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')

ИТОГОВАЯ СВОДКА
1) Использовано 1772 train и 318 test лабораторных меток; признаков 416.
2) Выбранная по CV модель RandomForest; test MAE=0.063, RMSE=0.07805, WAPE=30.04%.
3) Проверьте согласованность test-метрик с временной CV таблицей и графиком ошибок.
4) Перед производственным применением нужны технологическая валидация, проверка дрейфа датчиков и безопасный режим работы; виртуальный датчик — поддержка оператора, а не замена лабораторного контроля.
Out[0]: 284
